# Event Weights

## Overview

This notebook demonstrates the public functions in `event_weights` using triple-barrier events built from synthetic dollar bars.
- Problem: overlapping event horizons give repeated information too much influence during model fitting.
- Approach: measure event uniqueness, discount overlapping returns, and compare standard with sequential bootstrap samples.
- Concurrent Events: It counts the active event horizons on each bar.
- Event Weights: It visualizes uniqueness and time-decay weights as in AFML Figures 4.1 and 4.3.
- Sequential Bootstrap: It compares bootstrap uniqueness distributions as in AFML Figure 4.2.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.Series.iteritems = pd.Series.items
np.random.seed(7)

In [ ]:
from src.data_preprocessing.event_labeling import get_daily_volatility, get_events, get_vertical_barriers
from src.data_preprocessing.market_structured_bars import get_dollar_bars
from src.data_preprocessing.event_weights import (
    apply_time_decay,
    compute_average_uniqueness_weights,
    compute_return_attribution_weights,
    count_concurrent_events,
    generate_random_t1,
    run_monte_carlo_trial,
)


## Define Synthetic Trades And Build Events

This cell constructs triple-barrier events from the reproducible synthetic trade stream.
- Dollar bars provide the event-time close series used to define the price path.
- The labeling pipeline produces overlapping event horizons for the weighting examples.


In [ ]:
rng = np.random.default_rng(42)
trading_days = pd.bdate_range("2024-01-02", periods=20, tz="UTC")
trades_per_day = 48
minutes_from_open = np.tile(np.arange(trades_per_day) * 5 + 14 * 60 + 30, len(trading_days))
timestamps = trading_days.repeat(trades_per_day) + pd.to_timedelta(minutes_from_open, unit="m")
prices = 180.0 * np.exp(np.cumsum(rng.normal(0.00005, 0.0015, len(timestamps))))
sizes = rng.lognormal(3.7, 0.55, len(timestamps)).round().astype(int)
trades = pd.DataFrame({"timestamp": timestamps, "symbol": "AAPL", "price": prices, "size": sizes})
notional = trades["price"].astype(float) * trades["size"].astype(float)
target_bars_per_day = 24
trading_days = len(trading_days)
target_num_bars = max(1, trading_days * target_bars_per_day)
dollar_threshold = float(notional.sum() / target_num_bars)
dollar_bars = get_dollar_bars(trades, threshold=dollar_threshold).ohlcv
dollar_close = dollar_bars["close"].astype(float)

daily_volatility = get_daily_volatility(dollar_close, span0=50)
event_spacing = 5
t_events = pd.DatetimeIndex(dollar_close.index[::event_spacing])
t1 = get_vertical_barriers(t_events, dollar_close, num_bars=10)
targets = daily_volatility.reindex(t_events)
side = dollar_close.pct_change(5).reindex(t_events).apply(lambda value: 1.0 if value >= 0 else -1.0)

eligible_events = targets.dropna().index.intersection(side.dropna().index).intersection(t1.index)
events = get_events(
    close=dollar_close,
    t_events=eligible_events,
    pt_sl=[1.0, 1.0],
    trgt=targets,
    min_ret=float(targets.loc[eligible_events].quantile(0.25)),
    num_threads=1,
    t1=t1,
    side=side,
)

print("source: synthetic intraday trade stream")
print(f"trading_days: {trading_days}")
print(f"target_bars_per_day: {target_bars_per_day}")
print(f"num_dollar_bars: {len(dollar_close):,}")
print(f"num_events: {len(events):,}")
events.head()

## Count Concurrent Events

This cell counts the number of active event horizons at each dollar bar.
- count_concurrent_events provides the overlap series used by the weighting functions.
- Higher counts reduce the unique contribution of each overlapping event.


In [ ]:
num_concurrent_events = count_concurrent_events(
    close_idx=dollar_close.index,
    t1=events["t1"],
    molecule=events.index,
)

## Compute Uniqueness And Weights

This cell computes uniqueness, return-attribution, and time-decay weights.
- compute_average_uniqueness_weights produces the per-event uniqueness values shown in AFML Figure 4.1.
- compute_return_attribution_weights discounts returns that coincide with many active events.
- apply_time_decay produces the piecewise-linear curves shown in AFML Figure 4.3.


In [ ]:
average_uniqueness_weights = compute_average_uniqueness_weights(
    t1=events["t1"],
    num_co_events=num_concurrent_events,
    molecule=events.index,
)
event_weights = compute_return_attribution_weights(
    t1=events["t1"],
    num_co_events=num_concurrent_events,
    close=dollar_close,
    molecule=events.index,
)
time_decay_weights = apply_time_decay(event_weights, clf_last_w=0.5)

pd.concat(
    {
        "avg_uniqueness_weight": average_uniqueness_weights,
        "sample_weight": event_weights,
        "time_decay_weight": time_decay_weights,
    },
    axis=1,
).head()

This cell plots event uniqueness and the time-decay curves.


In [ ]:
weights_df = pd.DataFrame(
    {
        "avg_uniqueness_weight": average_uniqueness_weights,
        "sample_weight": event_weights,
        "time_decay_weight": time_decay_weights,
    }
)
time_decay_curves = pd.DataFrame(
    {
        f"c = {value:g}": apply_time_decay(event_weights, clf_last_w=value)
        for value in [1.0, 0.75, 0.5, 0.0, -0.25, -0.5]
    }
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(average_uniqueness_weights, bins=12, color="tab:blue", edgecolor="white")
axes[0].set_title("AFML Figure 4.1: Uniqueness values")
axes[0].set_xlabel("average uniqueness")
axes[0].set_ylabel("num events")
axes[0].grid(axis="y", alpha=0.25)

time_decay_curves.plot(ax=axes[1], lw=1.2)
axes[1].set_title("AFML Figure 4.3: Piecewise-linear time decay")
axes[1].set_xlabel("event start")
axes[1].set_ylabel("decay factor")
axes[1].grid(alpha=0.25)
fig.tight_layout()

weights_df.head()


## Sequential Bootstrap

This cell compares standard and sequential bootstrap samples through repeated simulated event horizons.
- run_monte_carlo_trial reports mean average uniqueness for one standard and one sequential-bootstrap sample.
- The paired histograms follow AFML Figure 4.2.


This cell generates repeated simulated event horizons and summarizes the two bootstrap methods.


In [ ]:
random_t1 = generate_random_t1(num_obs=10, num_bars=50, max_h=5)
monte_carlo = pd.DataFrame(
    [run_monte_carlo_trial(num_obs=25, num_bars=250, max_h=10) for _ in range(50)]
)

print("synthetic_horizons")
display(random_t1.head())
monte_carlo.agg(["mean", "std"])

This cell plots the standard and sequential-bootstrap uniqueness distributions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axes[0].hist(monte_carlo["std_u"], bins=12, color="tab:orange", edgecolor="white")
axes[0].set_title("Standard bootstrap")
axes[0].set_xlabel("mean average uniqueness")
axes[0].set_ylabel("num trials")
axes[0].grid(axis="y", alpha=0.25)

axes[1].hist(monte_carlo["seq_u"], bins=12, color="tab:green", edgecolor="white")
axes[1].set_title("Sequential bootstrap")
axes[1].set_xlabel("mean average uniqueness")
axes[1].grid(axis="y", alpha=0.25)

fig.suptitle("AFML Figure 4.2: Standard vs. sequential bootstrap")
fig.tight_layout()
